In [7]:
import pandas as pd
import itertools
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

In [8]:
INPUT_DIR = '../data/human_annotated_test_set'
df = pd.read_json(f'{INPUT_DIR}/human_test_set.jsonl', lines=True)
sub_df = pd.read_json(f'../data/submissions.jsonl', lines=True)
df['id'] = df['sub_id'].str.rsplit('_', n=1).str[0]

df = df.merge(
    sub_df[['sub_id', 'code', 'lang']],
    left_on='sub_id',
    right_on='sub_id',
    how='left'
)

In [ ]:
pairwise_records = []
criteria_cols = ['efficiency', 'correctness', 'readability']

for problem_id, group in df.groupby('id'):
    submissions = group.to_dict('records')
    
    for sub1, sub2 in itertools.combinations(submissions, 2):
        
        # Duyệt qua từng tiêu chí
        for criteria in criteria_cols:
            score_col = f'{criteria}_score'
            score1 = sub1[score_col]
            score2 = sub2[score_col]
            
            if score1 > score2:
                label = 1 
            elif score1 < score2:
                label = 0  
            else:
                label = 0.5  
                
            pairwise_records.append({
                'id': problem_id,
                'criteria': criteria,
                'sub_id_1': sub1['sub_id'],
                'sub_id_2': sub2['sub_id'],
                'label': label
            })

pairwise_df = pd.DataFrame(pairwise_records)
pairwise_df.to_json(f'{INPUT_DIR}/pairwise_human_set.jsonl', orient='records', lines=True)

In [10]:
unique_problems = pairwise_df['id'].unique()
train_problems, test_problems = train_test_split(unique_problems, test_size=0.2, random_state=42)

train_df = pairwise_df[pairwise_df['id'].isin(train_problems)].copy()
test_df = pairwise_df[pairwise_df['id'].isin(test_problems)].copy()

train_df.to_json(f'{INPUT_DIR}/pairwise_human_train.jsonl', orient='records', lines=True)
test_df.to_json(f'{INPUT_DIR}/pairwise_human_test.jsonl', orient='records', lines=True)